# KeelNet Stage 5: Support-Constrained Learning Comparison Template

Use this notebook as the Stage 5 working file for the official team workflow:

1. edit locally in VS Code
2. open this notebook in browser Google Colab
3. rerun the setup cell after pushing code changes
4. save artifacts to Google Drive or your local runtime project folder

This notebook keeps the Stage 5 notes, implementation hints, run commands, and shareable outputs in one place.


## Stage Notes

### Goal

Compare the strongest modular grounded-QA pipeline against a direct support-constrained learning objective under matched conditions.

### Scope

- input: the same grounded QA examples used in Stages 1 to 4
- output: answer or `ABSTAIN`
- comparison: strongest modular baseline versus support-constrained learner

### Main Change

- replace post-hoc-only control with a direct support-constrained learning objective

### Main Metrics

- overall `F1`
- answerable `F1`
- unsupported-answer rate
- supported-answer rate
- abstain `F1`

### What This Stage Validates

- changing the learning target adds value beyond modular tuning
- the gain is real under matched data, backbone, and evaluation conditions

### Handoff Condition

Do not move to the next stage until the support-constrained learner either clearly beats the strongest modular baseline or gives a crisp failure diagnosis you can explain.


<h2 style="color: #1d4ed8;">1. Setup And Sync</h2>

Run this cell in either hosted Google Colab or Google Colab connected to a local Jupyter runtime.

What it does:

- hosted Colab: mounts Drive, loads `HF_TOKEN` from Colab Secrets if available, clones or updates `/content/KeelNet`, checks out the stage branch, and installs the project
- local runtime: reuses your local repo, uses a local project folder for artifacts, reads `HF_TOKEN` from the environment if available, and installs the project into the current kernel environment

Path reminder:

- hosted Colab defaults: repo `/content/KeelNet`, project folder `/content/drive/MyDrive/KeelNet`
- local runtime defaults: repo `/content/KeelNet` if present, otherwise your current local checkout; project folder `/content/KeelNet-local`
- optional overrides: `KEELNET_REPO_DIR`, `KEELNET_PROJECT_DIR`, and `KEELNET_DRIVE_SYNC_DIR`

Reliability reminder:

- hosted Colab defaults to a fresh clone on each setup run to avoid stale or conflicted repo state
- set `FORCE_FRESH_CLONE = False` in the setup cell or `KEELNET_FORCE_FRESH_CLONE=0` if you intentionally want to reuse `/content/KeelNet`


In [1]:
from pathlib import Path
import os
import shutil
import subprocess
import sys


GIT_REPO_URL = "https://github.com/naufalkmd/KeelNet.git"
DEFAULT_GIT_BRANCH = 'main'
GIT_BRANCH = os.environ.get("KEELNET_GIT_BRANCH", DEFAULT_GIT_BRANCH)
HOSTED_COLAB_PROJECT_DIR = Path("/content/drive/MyDrive/KeelNet")
DEFAULT_LOCAL_PROJECT_DIR = Path("/content/KeelNet-local")
DEFAULT_LOCAL_REPO_DIR = Path("/content/KeelNet")
FORCE_FRESH_CLONE = True

env_force_fresh_clone = os.environ.get("KEELNET_FORCE_FRESH_CLONE")
if env_force_fresh_clone is not None:
    FORCE_FRESH_CLONE = env_force_fresh_clone.strip().lower() not in ('0', 'false', 'no')


def detect_runtime_mode() -> str:
    try:
        import google.colab  # noqa: F401
    except ImportError:
        return "local-runtime"
    return "hosted-colab"


RUNTIME_MODE = detect_runtime_mode()
IS_HOSTED_COLAB = RUNTIME_MODE == "hosted-colab"
PROJECT_STORAGE_LABEL = "Drive project dir" if IS_HOSTED_COLAB else "Local project dir"


def run_setup(cmd, *, cwd: Path | None = None) -> None:
    rendered = [str(part) for part in cmd]
    print("$", " ".join(rendered))
    subprocess.run(rendered, check=True, cwd=str(cwd) if cwd else None)


def configure_project_storage() -> Path:
    if IS_HOSTED_COLAB:
        from google.colab import drive

        project_dir = Path(os.environ.get("KEELNET_PROJECT_DIR", str(HOSTED_COLAB_PROJECT_DIR)))
        drive.mount("/content/drive", force_remount=False)
        if not str(project_dir).startswith("/content/drive/"):
            raise ValueError(
                f"KEELNET_PROJECT_DIR must point inside /content/drive in hosted Colab, got: {project_dir}"
            )
        project_dir.mkdir(parents=True, exist_ok=True)
        print(f"{PROJECT_STORAGE_LABEL}: {project_dir}")
        return project_dir

    project_dir = Path(os.environ.get("KEELNET_PROJECT_DIR", str(DEFAULT_LOCAL_PROJECT_DIR))).expanduser().resolve()
    project_dir.mkdir(parents=True, exist_ok=True)
    print(f"{PROJECT_STORAGE_LABEL}: {project_dir}")
    return project_dir


def configure_drive_project_dir(project_storage_dir: Path) -> Path | None:
    if IS_HOSTED_COLAB:
        print(f"Drive project dir: {project_storage_dir}")
        return project_storage_dir.resolve()

    env_drive_dir = os.environ.get("KEELNET_DRIVE_SYNC_DIR")
    if not env_drive_dir:
        print(
            "Drive project dir: disabled "
            "(set KEELNET_DRIVE_SYNC_DIR to a local Google Drive sync folder to mirror artifacts there)."
        )
        return None

    drive_project_dir = Path(env_drive_dir).expanduser().resolve()
    drive_project_dir.mkdir(parents=True, exist_ok=True)
    print(f"Drive project dir: {drive_project_dir}")
    return drive_project_dir


def configure_hf_token() -> None:
    if os.environ.get("HF_TOKEN"):
        print("HF_TOKEN already set in the environment.")
        return

    if not IS_HOSTED_COLAB:
        print("HF_TOKEN not set in the environment; continuing with anonymous HF access.")
        return

    try:
        from google.colab import userdata
    except ImportError:
        print("Colab secrets are unavailable; continuing without HF_TOKEN.")
        return

    try:
        token = userdata.get("HF_TOKEN")
    except Exception:
        token = None

    if token:
        os.environ["HF_TOKEN"] = token
        print("Loaded HF_TOKEN from Colab secrets.")
    else:
        print("HF_TOKEN not found in Colab secrets; continuing with anonymous HF access.")


def resolve_local_repo_dir() -> Path:
    candidates: list[Path] = []
    env_repo_dir = os.environ.get("KEELNET_REPO_DIR")
    if env_repo_dir:
        candidates.append(Path(env_repo_dir).expanduser())
    candidates.append(DEFAULT_LOCAL_REPO_DIR)
    cwd = Path.cwd().resolve()
    candidates.append(cwd)
    candidates.extend(cwd.parents)

    seen: set[Path] = set()
    for candidate in candidates:
        resolved = candidate.resolve()
        if resolved in seen:
            continue
        seen.add(resolved)
        if (resolved / ".git").exists() and (resolved / "pyproject.toml").exists():
            return resolved

    raise FileNotFoundError(
        "Could not find the KeelNet repo. Set KEELNET_REPO_DIR to your local checkout before running this cell."
    )


def ensure_repo() -> Path:
    if not IS_HOSTED_COLAB:
        return resolve_local_repo_dir()

    local_repo_dir = Path(os.environ.get("KEELNET_REPO_DIR", str(DEFAULT_LOCAL_REPO_DIR)))
    if FORCE_FRESH_CLONE and local_repo_dir.exists():
        os.chdir("/content")
        print(f"Removing existing hosted Colab repo for a fresh clone: {local_repo_dir}")
        shutil.rmtree(local_repo_dir)

    if (local_repo_dir / ".git").exists():
        run_setup(["git", "fetch", "origin"], cwd=local_repo_dir)
        run_setup(["git", "checkout", GIT_BRANCH], cwd=local_repo_dir)
        run_setup(["git", "pull", "--ff-only", "origin", GIT_BRANCH], cwd=local_repo_dir)
    else:
        run_setup(["git", "clone", "--branch", GIT_BRANCH, GIT_REPO_URL, str(local_repo_dir)])
    return local_repo_dir.resolve()


PROJECT_STORAGE_DIR = configure_project_storage()
DRIVE_PROJECT_DIR = configure_drive_project_dir(PROJECT_STORAGE_DIR)
configure_hf_token()
REPO_DIR = ensure_repo().resolve()
os.chdir(REPO_DIR)
print(f"Runtime mode: {RUNTIME_MODE}")
print(f"Runtime repo dir: {REPO_DIR}")
print(f"Force fresh clone: {FORCE_FRESH_CLONE}")
CURRENT_BRANCH = subprocess.run(
    ["git", "rev-parse", "--abbrev-ref", "HEAD"],
    check=True,
    cwd=REPO_DIR,
    capture_output=True,
    text=True,
).stdout.strip()
print(f"Git branch: {CURRENT_BRANCH}")
run_setup([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_DIR)])


def mirror_output_root(output_root: Path) -> Path | None:
    if DRIVE_PROJECT_DIR is None:
        print("Drive artifact mirror is disabled for this runtime.")
        return None

    output_root = Path(output_root).expanduser().resolve()
    if not output_root.exists():
        print(f"Nothing to mirror yet: {output_root}")
        return None

    drive_project_dir = DRIVE_PROJECT_DIR.expanduser().resolve()
    try:
        relative_output = output_root.relative_to(PROJECT_STORAGE_DIR.expanduser().resolve())
    except ValueError:
        relative_output = Path("artifacts") / output_root.name
    drive_output_root = drive_project_dir / relative_output

    if output_root == drive_output_root:
        print(f"Artifacts already stored in Drive: {output_root}")
        return output_root

    drive_output_root.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(output_root, drive_output_root, dirs_exist_ok=True)
    print(f"Mirrored artifacts to Drive: {drive_output_root}")
    return drive_output_root


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive project dir: /content/drive/MyDrive/KeelNet
Drive project dir: /content/drive/MyDrive/KeelNet
Loaded HF_TOKEN from Colab secrets.
Removing existing hosted Colab repo for a fresh clone: /content/KeelNet
$ git clone --branch main https://github.com/naufalkmd/KeelNet.git /content/KeelNet
Runtime mode: hosted-colab
Runtime repo dir: /content/KeelNet
Force fresh clone: True
Git branch: main
$ /usr/bin/python3 -m pip install -q -e /content/KeelNet


<h2 style="color: #1d4ed8;">2. Configure The Run</h2>

Set `AUTHOR_NAME` to your name. This notebook builds the stage-specific `RUN_NAME` automatically as `yourname-stage5-v1`, `yourname-stage5-v2`, `yourname-stage5-v3`, and so on based on completed runs.

This notebook first tries to auto-fill `MODULAR_BASELINE_EVAL_PATH` from the latest completed Stage 4 run under the current artifact root. Override it if you want to compare against a different controller baseline.

Review `TARGET_METRICS`, `SUGGESTED_MODULES`, `SMOKE_TEST_COMMANDS`, and `STAGE_COMMANDS` before you move into implementation.

This cell also prints the important values you should check before running stage commands.

It creates a small `RUN_STARTED.txt` file in the current run folder immediately, so you can confirm the output path is correct before training or evaluation finishes.


In [2]:
from pathlib import Path
import json
import re
import subprocess
import sys

import torch

REPO_DIR = Path(REPO_DIR).resolve()
PROJECT_STORAGE_DIR = Path(PROJECT_STORAGE_DIR).resolve()
DRIVE_PROJECT_DIR = Path(DRIVE_PROJECT_DIR).resolve() if DRIVE_PROJECT_DIR is not None else None

AUTHOR_NAME = "naufal"
RUN_BASENAME = f"{AUTHOR_NAME}-stage5"
ARTIFACTS_ROOT = PROJECT_STORAGE_DIR / "artifacts" / "stage5_colab"
COMPLETION_MARKER_NAME = "RUN_COMPLETED.txt"

STAGE_TITLE = "Stage 5: Support-Constrained Learning Comparison"
STAGE_OBJECTIVE = "Compare the best modular pipeline against a direct support-constrained learning objective under matched conditions."
TARGET_METRICS = [
    "overall F1",
    "answerable F1",
    "unsupported-answer rate",
    "supported-answer rate",
    "abstain F1",
]
IMPLEMENTATION_HINTS = [
    "input: the same grounded QA split used in Stages 1 to 4",
    "output: a jointly trained answer-support-abstain learner",
    "compare directly against the strongest modular Stage 4 baseline when available",
]
SUGGESTED_MODULES = ["keelnet.learn", "keelnet.metrics", "keelnet.train", "keelnet.evaluate"]


def completed_versions(root: Path, run_basename: str) -> list[int]:
    versions: list[int] = []
    if not root.exists():
        return versions

    pattern = re.compile(rf"^{re.escape(run_basename)}-v(\d+)$")
    for child in root.iterdir():
        if not child.is_dir():
            continue
        match = pattern.match(child.name)
        if match and (child / COMPLETION_MARKER_NAME).exists():
            versions.append(int(match.group(1)))
    return sorted(versions)


def completed_run_dirs(root: Path, *, run_prefix: str | None = None) -> list[Path]:
    if not root.exists():
        return []

    pattern = re.compile(rf"^{re.escape(run_prefix)}-v(\d+)$") if run_prefix is not None else None
    runs: list[Path] = []
    for child in root.iterdir():
        if not child.is_dir():
            continue
        if not (child / COMPLETION_MARKER_NAME).exists():
            continue
        if pattern is not None and not pattern.match(child.name):
            continue
        runs.append(child)
    return sorted(runs, key=lambda path: (path.stat().st_mtime, path.name))


def default_upstream_path(stage_folder: str, relative_path: str, *, preferred_run_prefix: str | None = None) -> str | None:
    stage_root = PROJECT_STORAGE_DIR / "artifacts" / stage_folder
    ordered_runs: list[Path] = []

    if preferred_run_prefix is not None:
        ordered_runs.extend(reversed(completed_run_dirs(stage_root, run_prefix=preferred_run_prefix)))

    for run_dir in reversed(completed_run_dirs(stage_root)):
        if run_dir not in ordered_runs:
            ordered_runs.append(run_dir)

    relative = Path(relative_path)
    for run_dir in ordered_runs:
        candidate = run_dir / relative
        if candidate.exists():
            return str(candidate)
    return None


RUN_VERSION = max(completed_versions(ARTIFACTS_ROOT, RUN_BASENAME), default=0) + 1
RUN_NAME = f"{RUN_BASENAME}-v{RUN_VERSION}"
OUTPUT_ROOT = ARTIFACTS_ROOT / RUN_NAME
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
RUN_MARKER_FILE = OUTPUT_ROOT / "RUN_STARTED.txt"
RUN_NOTES_FILE = OUTPUT_ROOT / "run-notes-template.md"
RUN_SUMMARY_FILE = OUTPUT_ROOT / "run-summary.json"
COMPLETION_MARKER_FILE = OUTPUT_ROOT / COMPLETION_MARKER_NAME

NOTEBOOK_ARCHIVE_DIR = OUTPUT_ROOT / "executed-notebook"
NOTEBOOK_ARCHIVE_DIR.mkdir(parents=True, exist_ok=True)
EXECUTED_NOTEBOOK_PATH = NOTEBOOK_ARCHIVE_DIR / f"{RUN_NAME}-executed.ipynb"
EXECUTED_NOTEBOOK_INSTRUCTIONS_FILE = NOTEBOOK_ARCHIVE_DIR / "README-save-executed-notebook.txt"
EXECUTED_NOTEBOOK_INSTRUCTIONS_FILE.write_text(
    "\n".join(
        [
            "Keep the canonical stage notebook in the repo as a clean template.",
            "",
            "The save/share cells try to archive the live notebook automatically when Colab exposes it.",
            "",
            "If automatic capture is unavailable, preserve a meaningful run with outputs manually:",
            "1. in Colab, use File -> Download .ipynb or File -> Save a copy in Drive",
            f"2. save the exported file as: {EXECUTED_NOTEBOOK_PATH.name}",
            f"3. place that file inside: {NOTEBOOK_ARCHIVE_DIR}",
            "",
            "Anything saved in this run folder stays outside the template-sync path and will mirror to Drive when KEELNET_DRIVE_SYNC_DIR is enabled.",
        ]
    )
    + "\n",
    encoding="utf-8",
)
AUTO_SAVED_EXECUTED_NOTEBOOK_PATH = None


def save_executed_notebook_snapshot() -> Path | None:
    global AUTO_SAVED_EXECUTED_NOTEBOOK_PATH

    try:
        from google.colab import _message
    except ImportError:
        print(
            "Automatic executed-notebook capture is unavailable in this runtime. "
            f"If you want the notebook archive, save it manually to {EXECUTED_NOTEBOOK_PATH}."
        )
        return None

    try:
        response = _message.blocking_request("get_ipynb", timeout_sec=30)
    except TypeError:
        response = _message.blocking_request("get_ipynb", request="", timeout_sec=30)
    except Exception as exc:
        print(
            "Automatic executed-notebook capture failed. "
            f"Save the notebook manually to {EXECUTED_NOTEBOOK_PATH}. Error: {exc}"
        )
        return None

    notebook = response.get("ipynb") if isinstance(response, dict) else None
    if notebook is None:
        print(
            "Automatic executed-notebook capture did not return notebook content. "
            f"Save the notebook manually to {EXECUTED_NOTEBOOK_PATH}."
        )
        return None

    metadata = notebook.setdefault("metadata", {})
    keelnet_metadata = metadata.setdefault("keelnet", {})
    keelnet_metadata["run_name"] = RUN_NAME
    keelnet_metadata["executed_notebook_target"] = str(EXECUTED_NOTEBOOK_PATH)
    source_notebook_name = response.get("path") if isinstance(response, dict) else None
    if source_notebook_name:
        keelnet_metadata["source_notebook_name"] = source_notebook_name

    EXECUTED_NOTEBOOK_PATH.write_text(
        json.dumps(notebook, indent=1, ensure_ascii=False) + "\n",
        encoding="utf-8",
    )
    AUTO_SAVED_EXECUTED_NOTEBOOK_PATH = EXECUTED_NOTEBOOK_PATH
    print(f"Saved executed notebook snapshot: {EXECUTED_NOTEBOOK_PATH}")
    return EXECUTED_NOTEBOOK_PATH

DEFAULT_STAGE4_CONTROL_EVAL = default_upstream_path(
    "stage4_colab",
    "control_eval.json",
    preferred_run_prefix=f"{AUTHOR_NAME}-stage4",
)
MODULAR_BASELINE_EVAL_PATH = DEFAULT_STAGE4_CONTROL_EVAL

MODEL_NAME = "distilbert-base-uncased"
NUM_TRAIN_EPOCHS = 3.0
TRAIN_BATCH_SIZE = 8
EVAL_BATCH_SIZE = 8
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.0
KEEP_LOSS_WEIGHT = 1.0
SUPPORT_LOSS_WEIGHT = 1.0
KEEP_POSITIVE_WEIGHT = 1.0
KEEP_NEGATIVE_WEIGHT = 2.0
MAX_TRAIN_SAMPLES = None
MAX_EVAL_SAMPLES = None
MAX_UNSUPPORTED_ANSWER_RATE = 20.0
KEEP_THRESHOLD_MIN = 0.05
KEEP_THRESHOLD_MAX = 0.95
KEEP_THRESHOLD_STEP = 0.05

RUN_SMOKE_TEST = False
SMOKE_TEST_TRAIN_SAMPLES = 256
SMOKE_TEST_EVAL_SAMPLES = 128
SMOKE_TEST_EPOCHS = 1.0

LEARNER_DIR = OUTPUT_ROOT / "learner"
LEARNER_EVAL = OUTPUT_ROOT / "learner_eval.json"

if MODULAR_BASELINE_EVAL_PATH is not None:
    MODULAR_BASELINE_EVAL_PATH = Path(MODULAR_BASELINE_EVAL_PATH).expanduser().resolve()
    if not MODULAR_BASELINE_EVAL_PATH.exists():
        raise FileNotFoundError(f"Baseline eval file not found: {MODULAR_BASELINE_EVAL_PATH}")


def maybe_add_arg(cmd: list[str], flag: str, value: object | None) -> None:
    if value is None:
        return
    cmd.extend([flag, str(value)])


def build_train_command(
    output_dir: Path,
    *,
    max_train_samples: int | None,
    max_eval_samples: int | None,
    num_train_epochs: float,
) -> list[str]:
    cmd = [
        sys.executable,
        "-m",
        "keelnet.learn",
        "train",
        "--output-dir",
        str(output_dir),
        "--model-name",
        MODEL_NAME,
        "--num-train-epochs",
        str(num_train_epochs),
        "--train-batch-size",
        str(TRAIN_BATCH_SIZE),
        "--eval-batch-size",
        str(EVAL_BATCH_SIZE),
        "--learning-rate",
        str(LEARNING_RATE),
        "--weight-decay",
        str(WEIGHT_DECAY),
        "--warmup-ratio",
        str(WARMUP_RATIO),
        "--keep-loss-weight",
        str(KEEP_LOSS_WEIGHT),
        "--support-loss-weight",
        str(SUPPORT_LOSS_WEIGHT),
        "--keep-positive-weight",
        str(KEEP_POSITIVE_WEIGHT),
        "--keep-negative-weight",
        str(KEEP_NEGATIVE_WEIGHT),
    ]
    maybe_add_arg(cmd, "--max-train-samples", max_train_samples)
    maybe_add_arg(cmd, "--max-eval-samples", max_eval_samples)
    return cmd


def build_eval_command(
    model_path: Path,
    output_path: Path,
    *,
    max_eval_samples: int | None,
) -> list[str]:
    cmd = [
        sys.executable,
        "-m",
        "keelnet.learn",
        "evaluate",
        "--model-path",
        str(model_path),
        "--output-path",
        str(output_path),
        "--eval-batch-size",
        str(EVAL_BATCH_SIZE),
        "--keep-threshold-min",
        str(KEEP_THRESHOLD_MIN),
        "--keep-threshold-max",
        str(KEEP_THRESHOLD_MAX),
        "--keep-threshold-step",
        str(KEEP_THRESHOLD_STEP),
        "--max-unsupported-answer-rate",
        str(MAX_UNSUPPORTED_ANSWER_RATE),
    ]
    maybe_add_arg(cmd, "--max-eval-samples", max_eval_samples)
    return cmd


smoke_model_dir = OUTPUT_ROOT / "smoke-learner"
smoke_eval_path = OUTPUT_ROOT / "smoke-learner-eval.json"
smoke_train_command = build_train_command(
    smoke_model_dir,
    max_train_samples=SMOKE_TEST_TRAIN_SAMPLES,
    max_eval_samples=SMOKE_TEST_EVAL_SAMPLES,
    num_train_epochs=SMOKE_TEST_EPOCHS,
)
smoke_eval_command = build_eval_command(
    smoke_model_dir,
    smoke_eval_path,
    max_eval_samples=SMOKE_TEST_EVAL_SAMPLES,
)
stage_train_command = build_train_command(
    LEARNER_DIR,
    max_train_samples=MAX_TRAIN_SAMPLES,
    max_eval_samples=MAX_EVAL_SAMPLES,
    num_train_epochs=NUM_TRAIN_EPOCHS,
)
stage_eval_command = build_eval_command(
    LEARNER_DIR,
    LEARNER_EVAL,
    max_eval_samples=MAX_EVAL_SAMPLES,
)

SMOKE_TEST_COMMANDS = [smoke_train_command, smoke_eval_command]
STAGE_COMMANDS = [stage_train_command, stage_eval_command]

RUN_MARKER_FILE.write_text(
    "\n".join(
        [
            f"stage={STAGE_TITLE}",
            f"run_name={RUN_NAME}",
            f"run_version=v{RUN_VERSION}",
            f"runtime_mode={RUNTIME_MODE}",
            f"repo_dir={REPO_DIR}",
            f"project_storage_dir={PROJECT_STORAGE_DIR}",
            f"git_branch={CURRENT_BRANCH}",
            f"baseline_eval_path={MODULAR_BASELINE_EVAL_PATH}",
            "status=configured",
            "note=This file is created when the config cell runs.",
        ]
    )
    + "\n",
    encoding="utf-8",
)

print(f"Runtime mode: {RUNTIME_MODE}")
print(f"Repo dir: {REPO_DIR}")
print(f"{PROJECT_STORAGE_LABEL}: {PROJECT_STORAGE_DIR}")
print(f"Drive project dir: {DRIVE_PROJECT_DIR}")
print(f"Artifacts root: {ARTIFACTS_ROOT}")
print(f"Run basename: {RUN_BASENAME}")
print(f"Run version: v{RUN_VERSION}")
print(f"Run output dir: {OUTPUT_ROOT}")
print(f"Executed notebook target: {EXECUTED_NOTEBOOK_PATH}")
print(f"Run marker file: {RUN_MARKER_FILE}")
print(f"Baseline eval path: {MODULAR_BASELINE_EVAL_PATH}")
print(f"Learner dir: {LEARNER_DIR}")
print(f"Learner eval file: {LEARNER_EVAL}")
print(f"Model name: {MODEL_NAME}")
print(f"Max unsupported-answer rate: {MAX_UNSUPPORTED_ANSWER_RATE}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print("Target metrics:", ", ".join(TARGET_METRICS))
print("Suggested modules:", ", ".join(SUGGESTED_MODULES))


def run(cmd):
    rendered = [str(part) for part in cmd]
    print("$", " ".join(rendered))
    with subprocess.Popen(
        rendered,
        cwd=REPO_DIR,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        bufsize=1,
    ) as process:
        if process.stdout is not None:
            for line in process.stdout:
                print(line, end="", flush=True)
        return_code = process.wait()
    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, rendered)


def run_many(commands, *, label: str) -> bool:
    if not commands:
        print(f"No commands configured for {label} yet.")
        return False

    for index, cmd in enumerate(commands, start=1):
        print(f"\n[{label} {index}/{len(commands)}]")
        run(cmd)
    return True


Runtime mode: hosted-colab
Repo dir: /content/KeelNet
Drive project dir: /content/drive/MyDrive/KeelNet
Drive project dir: /content/drive/MyDrive/KeelNet
Artifacts root: /content/drive/MyDrive/KeelNet/artifacts/stage5_colab
Run basename: naufal-stage5
Run version: v1
Run output dir: /content/drive/MyDrive/KeelNet/artifacts/stage5_colab/naufal-stage5-v1
Executed notebook target: /content/drive/MyDrive/KeelNet/artifacts/stage5_colab/naufal-stage5-v1/executed-notebook/naufal-stage5-v1-executed.ipynb
Run marker file: /content/drive/MyDrive/KeelNet/artifacts/stage5_colab/naufal-stage5-v1/RUN_STARTED.txt
Baseline eval path: None
Learner dir: /content/drive/MyDrive/KeelNet/artifacts/stage5_colab/naufal-stage5-v1/learner
Learner eval file: /content/drive/MyDrive/KeelNet/artifacts/stage5_colab/naufal-stage5-v1/learner_eval.json
Model name: distilbert-base-uncased
Max unsupported-answer rate: 20.0
CUDA available: True
GPU: NVIDIA A100-SXM4-80GB
Target metrics: overall F1, answerable F1, unsuppor

<h2 style="color: #1d4ed8;">3. Validate The Environment</h2>

Run the project tests before stage-specific work. This confirms the installed code path is at least minimally healthy inside the current runtime.


In [3]:
from pathlib import Path
import re


CONFLICT_MARKER_PATTERN = re.compile(r"^(<<<<<<<|=======|>>>>>>>)", re.MULTILINE)


def _problem_excerpt(path: Path, *, marker_line: int, context_lines: int = 2) -> str:
    lines = path.read_text(encoding="utf-8").splitlines()
    start = max(1, marker_line - context_lines)
    end = min(len(lines), marker_line + context_lines)
    excerpt_lines = ["--- Start of problematic section ---"]
    for line_number in range(start, end + 1):
        excerpt_lines.append(f"{line_number}: {lines[line_number - 1]}")
    excerpt_lines.append("--- End of problematic section ---")
    return "\n".join(excerpt_lines)


def assert_repo_has_no_conflict_markers(repo_dir: Path) -> None:
    candidate_paths: list[Path] = []
    for relative_path in ("pyproject.toml",):
        candidate = repo_dir / relative_path
        if candidate.exists():
            candidate_paths.append(candidate)

    for subdir in ("src", "tests"):
        root = repo_dir / subdir
        if root.exists():
            candidate_paths.extend(sorted(root.rglob("*.py")))

    for path in candidate_paths:
        try:
            text = path.read_text(encoding="utf-8")
        except UnicodeDecodeError:
            continue

        match = CONFLICT_MARKER_PATTERN.search(text)
        if match is None:
            continue

        marker_line = text.count("\n", 0, match.start()) + 1
        print(
            f"ERROR: Found potential Git merge conflict markers in {path} around line {marker_line}:"
        )
        print(_problem_excerpt(path, marker_line=marker_line))
        raise RuntimeError(
            "Please resolve these Git merge conflicts in the file directly, "
            "or rerun the setup cell to refresh /content/KeelNet, then rerun this cell."
        )


assert_repo_has_no_conflict_markers(REPO_DIR)
run([sys.executable, "-m", "unittest", "discover", "-s", str(REPO_DIR / "tests")])


$ /usr/bin/python3 -m unittest discover -s /content/KeelNet/tests
...................
----------------------------------------------------------------------
Ran 19 tests in 0.041s

OK


<h2 style="color: #1d4ed8;">4. Optional Smoke Test</h2>

Use this cell only after you fill in `SMOKE_TEST_COMMANDS` in the config cell. Keep those commands tiny so you can catch path, dependency, and runtime issues before a full Stage 5 run.


In [4]:
if RUN_SMOKE_TEST:
    ran_smoke = run_many(SMOKE_TEST_COMMANDS, label="smoke test")
    if not ran_smoke:
        print("Add one or more small commands to SMOKE_TEST_COMMANDS in the config cell.")
else:
    print("Smoke test skipped. Set RUN_SMOKE_TEST = True after you define SMOKE_TEST_COMMANDS.")



Smoke test skipped. Set RUN_SMOKE_TEST = True after you define SMOKE_TEST_COMMANDS.


<div style="border-left: 6px solid #c2410c; background: #fff7ed; padding: 12px 16px; border-radius: 8px;">
<strong>Implementation Starts Here</strong><br/>
Sections 1-4 are setup and validation. Section 5 onward is the main Stage 5 work area.
<ul>
  <li><strong>Start here:</strong> review the auto-built <code>keelnet.learn</code> train and evaluate commands, then compare the resulting learner against the strongest modular Stage 4 baseline.</li>
  <li><strong>Finish here:</strong> the new learning objective beats or clearly sharpens the best modular baseline under the same metrics and data split.</li>
  <li><strong>Out of scope:</strong> retrieval realism, adaptive balancing, open-ended generation claims.</li>
</ul>
</div>


## IMPLEMENTATION 1: Run The Stage 5 Constrained-Learning Comparison

This is the main Stage 5 implementation and run section. Everything before this point is setup, sync, or validation.

The config cell now builds the default Stage 5 train and evaluate commands automatically. Review those commands, adjust the hyperparameters if needed, and then run this section.


In [5]:
run_many(STAGE_COMMANDS, label="stage command")


Streaming output truncated to the last 5000 lines.
 85%|████████▍ | 37699/44472 [42:24<07:32, 14.98it/s]
                                                     

 85%|████████▍ | 37749/44472 [42:28<07:31, 14.90it/s]
                                                     

 85%|████████▍ | 37799/44472 [42:31<07:31, 14.78it/s]
                                                     

 85%|████████▌ | 37849/44472 [42:34<07:27, 14.79it/s]
                                                     

 85%|████████▌ | 37899/44472 [42:38<07:21, 14.87it/s]
                                                     

 85%|████████▌ | 37949/44472 [42:41<07:21, 14.79it/s]
                                                     

 85%|████████▌ | 37999/44472 [42:44<07:14, 14.88it/s]
                                                     

 86%|████████▌ | 38049/44472 [42:48<07:18, 14.64it/s]
                                                     

 86%|████████▌ | 38099/44472 [42:51<07:09, 14.83it/s]
                       

True

## Stage Note Template

Keep your stage notes inside this notebook flow. Update them at three points:

1. before implementation: fill in the goal, success condition, and planned commands
2. after smoke test: record environment issues and command fixes
3. after a meaningful run: record metrics, verdict, and next actions

Use this structure for the generated run note:

- Run info
- Executed notebook archive target
- Goal
- Commands
- Main metrics
- What changed
- What worked
- What failed or looks risky
- Error cases to review
- Decision
- Next actions


<h2 style="color: #15803d;">Save Notes And Review Artifacts</h2>

This cell creates teammate-friendly note files inside the current run folder and lists the current artifacts. Update the generated notes as you learn what does and does not work in Stage 5: Support-Constrained Learning Comparison.


In [6]:
captured_notebook_path = save_executed_notebook_snapshot()
mirrored_output_root = mirror_output_root(OUTPUT_ROOT)

if not RUN_NOTES_FILE.exists():
    metric_lines = [f"- {metric}" for metric in TARGET_METRICS]
    RUN_NOTES_FILE.write_text(
        "\n".join(
            [
                f"# {STAGE_TITLE} Notes",
                "",
                "Update this note three times:",
                "1. before implementation: goal, success condition, and commands",
                "2. after smoke test: environment issues and command fixes",
                "3. after a meaningful run: metrics, verdict, and next actions",
                "",
                "## Run Info",
                f"- Branch: {CURRENT_BRANCH}",
                f"- `RUN_NAME`: {RUN_NAME}",
                f"- Output folder: {OUTPUT_ROOT}",
                f"- Executed notebook archive target: {EXECUTED_NOTEBOOK_PATH}",
                f"- Runtime mode: {RUNTIME_MODE}",
                "",
                "## Goal",
                f"- One-sentence objective: {STAGE_OBJECTIVE}",
                "- Success condition:",
                "- Out of scope:",
                "",
                "## Commands",
                f"- Smoke test command(s): {SMOKE_TEST_COMMANDS}",
                f"- Main command(s): {STAGE_COMMANDS}",
                f"- Modular baseline eval path: {MODULAR_BASELINE_EVAL_PATH}",
                f"- Output files to inspect: {LEARNER_EVAL}",
                "",
                "## Main Metrics",
                *metric_lines,
                "",
                "## What Changed",
                "- ",
                "",
                "## What Worked",
                "- ",
                "",
                "## What Failed Or Looks Risky",
                "- ",
                "",
                "## Error Cases To Review",
                "- ",
                "",
                "## Decision",
                "- Keep, revise, or stop:",
                "- Reason:",
                "",
                "## Next Actions",
                "1. ",
                "2. ",
                "3. ",
            ]
        )
        + "\n",
        encoding="utf-8",
    )

RUN_SUMMARY_FILE.write_text(
    json.dumps(
        {
            "stage": STAGE_TITLE,
            "run_name": RUN_NAME,
            "runtime_mode": RUNTIME_MODE,
            "git_branch": CURRENT_BRANCH,
            "output_root": str(OUTPUT_ROOT),
            "drive_project_dir": str(DRIVE_PROJECT_DIR) if DRIVE_PROJECT_DIR is not None else None,
            "mirrored_output_root": str(mirrored_output_root) if mirrored_output_root is not None else None,
            "executed_notebook_dir": str(NOTEBOOK_ARCHIVE_DIR),
            "executed_notebook_target": str(EXECUTED_NOTEBOOK_PATH),
            "executed_notebook_saved": captured_notebook_path is not None,
            "executed_notebook_instructions_file": str(EXECUTED_NOTEBOOK_INSTRUCTIONS_FILE),
            "learner_dir": str(LEARNER_DIR),
            "learner_eval": str(LEARNER_EVAL),
            "baseline_eval_path": str(MODULAR_BASELINE_EVAL_PATH) if MODULAR_BASELINE_EVAL_PATH is not None else None,
            "target_metrics": TARGET_METRICS,
            "suggested_modules": SUGGESTED_MODULES,
            "max_unsupported_answer_rate": MAX_UNSUPPORTED_ANSWER_RATE,
        },
        indent=2,
    )
    + "\n",
    encoding="utf-8",
)

print(f"Notes template: {RUN_NOTES_FILE}")
print(f"Run summary: {RUN_SUMMARY_FILE}")
print(f"Executed notebook target: {EXECUTED_NOTEBOOK_PATH}")
if captured_notebook_path is None:
    print(
        "Automatic notebook capture was unavailable. "
        f"If you want the notebook archive, save it manually to {EXECUTED_NOTEBOOK_PATH}."
    )
if mirrored_output_root is not None:
    print(f"Drive mirror: {mirrored_output_root}")
print("Current files under OUTPUT_ROOT:")
for path in sorted(OUTPUT_ROOT.rglob("*")):
    print(path)


Artifacts already stored in Drive: /content/drive/MyDrive/KeelNet/artifacts/stage5_colab/naufal-stage5-v1
Notes template: /content/drive/MyDrive/KeelNet/artifacts/stage5_colab/naufal-stage5-v1/run-notes-template.md
Run summary: /content/drive/MyDrive/KeelNet/artifacts/stage5_colab/naufal-stage5-v1/run-summary.json
Executed notebook target: /content/drive/MyDrive/KeelNet/artifacts/stage5_colab/naufal-stage5-v1/executed-notebook/naufal-stage5-v1-executed.ipynb
To preserve outputs, export the current notebook from Colab and save it at /content/drive/MyDrive/KeelNet/artifacts/stage5_colab/naufal-stage5-v1/executed-notebook/naufal-stage5-v1-executed.ipynb.
Drive mirror: /content/drive/MyDrive/KeelNet/artifacts/stage5_colab/naufal-stage5-v1
Current files under OUTPUT_ROOT:
/content/drive/MyDrive/KeelNet/artifacts/stage5_colab/naufal-stage5-v1/RUN_STARTED.txt
/content/drive/MyDrive/KeelNet/artifacts/stage5_colab/naufal-stage5-v1/executed-notebook
/content/drive/MyDrive/KeelNet/artifacts/stage

<h2 style="color: #15803d;">Final Check</h2>

Stage 5 is not successful just because a new objective trains.

Check all three:

- the constrained-learning variant is compared against the strongest modular baseline under matched conditions
- unsupported answers go down without collapsing answer quality through trivial refusal
- the gain comes from the new objective itself, not just threshold changes hidden inside the comparison

After that, document whether the next bottleneck is objective design, representation quality, or later adaptive balancing.


<h2 style="color: #15803d;">Share This Run</h2>

This cell prints a minimal share-ready summary for teammates, saves it into the current run folder, and marks the run as completed so the next run becomes the next version.

Update the metric lines after you review the outputs for this stage.


In [7]:
from datetime import datetime, timezone

captured_notebook_path = save_executed_notebook_snapshot()
mirrored_output_root = mirror_output_root(OUTPUT_ROOT)
metric_lines = [f"- {metric}: <fill in after review>" for metric in TARGET_METRICS]
if LEARNER_EVAL.exists():
    learner_results = json.loads(LEARNER_EVAL.read_text(encoding="utf-8"))
    dev_metrics = learner_results["dev_metrics"]
    dev_mix = learner_results["dev_mix"]
    metric_lines = [
        f"- Overall F1 (dev): {dev_metrics['overall_f1']:.2f}",
        f"- Answerable F1 (dev): {dev_metrics['answerable_f1']:.2f}",
        f"- Unsupported-answer rate (dev): {dev_metrics['unsupported_answer_rate']:.2f}",
        f"- Abstain F1 (dev): {dev_metrics['abstain_f1']:.2f}",
        f"- Supported-answer rate (among answers, dev): {dev_mix['supported_answer_rate']:.2f}",
        f"- Answer rate (dev): {dev_mix['answer_rate']:.2f}",
        f"- Selected keep threshold: {learner_results['selected_keep_threshold']:.2f}",
        f"- Max unsupported-answer rate budget: {learner_results['max_unsupported_answer_rate']:.2f}",
    ]
    if MODULAR_BASELINE_EVAL_PATH is not None and MODULAR_BASELINE_EVAL_PATH.exists():
        baseline_results = json.loads(MODULAR_BASELINE_EVAL_PATH.read_text(encoding="utf-8"))
        baseline_metrics = baseline_results.get("control_dev_metrics")
        baseline_mix = baseline_results.get("control_dev_mix")
        if baseline_metrics is not None and baseline_mix is not None:
            metric_lines = [
                f"- Overall F1 (dev): {baseline_metrics['overall_f1']:.2f} -> {dev_metrics['overall_f1']:.2f}",
                f"- Answerable F1 (dev): {baseline_metrics['answerable_f1']:.2f} -> {dev_metrics['answerable_f1']:.2f}",
                f"- Unsupported-answer rate (dev): {baseline_metrics['unsupported_answer_rate']:.2f} -> {dev_metrics['unsupported_answer_rate']:.2f}",
                f"- Abstain F1 (dev): {baseline_metrics['abstain_f1']:.2f} -> {dev_metrics['abstain_f1']:.2f}",
                f"- Supported-answer rate (among answers, dev): {baseline_mix['supported_answer_rate']:.2f} -> {dev_mix['supported_answer_rate']:.2f}",
                f"- Answer rate (dev): {baseline_mix['answer_rate']:.2f} -> {dev_mix['answer_rate']:.2f}",
                f"- Selected keep threshold: {learner_results['selected_keep_threshold']:.2f}",
                f"- Max unsupported-answer rate budget: {learner_results['max_unsupported_answer_rate']:.2f}",
            ]

share_lines = [
    f"# {STAGE_TITLE} Share Note",
    "",
    f"- runtime mode: {RUNTIME_MODE}",
    f"- branch name: {CURRENT_BRANCH}",
    f"- RUN_NAME: {RUN_NAME}",
    *metric_lines,
    f"- Output folder path: {OUTPUT_ROOT}",
    f"- Executed notebook archive target: {EXECUTED_NOTEBOOK_PATH}",
]
if mirrored_output_root is not None:
    share_lines.append(f"- Drive mirror path: {mirrored_output_root}")
share_note = "\n".join(share_lines) + "\n"
SHARE_NOTE_FILE = OUTPUT_ROOT / "collab-share-note.md"
SHARE_NOTE_FILE.write_text(share_note, encoding="utf-8")
COMPLETION_MARKER_FILE.write_text(
    "\n".join(
        [
            f"run_name={RUN_NAME}",
            f"completed_at={datetime.now(timezone.utc).isoformat()}",
            f"share_note={SHARE_NOTE_FILE.name}",
            "status=completed",
        ]
    )
    + "\n",
    encoding="utf-8",
)
print(share_note)
if captured_notebook_path is None:
    print(
        "Automatic notebook capture was unavailable. "
        f"If you want the notebook archive, save it manually to {EXECUTED_NOTEBOOK_PATH}."
    )
print(f"Saved share note: {SHARE_NOTE_FILE}")
print(f"Saved completion marker: {COMPLETION_MARKER_FILE}")
if mirrored_output_root is not None:
    mirror_output_root(OUTPUT_ROOT)


Artifacts already stored in Drive: /content/drive/MyDrive/KeelNet/artifacts/stage5_colab/naufal-stage5-v1
# Stage 5: Support-Constrained Learning Comparison Share Note

- runtime mode: hosted-colab
- branch name: main
- RUN_NAME: naufal-stage5-v1
- Overall F1 (dev): 68.28
- Answerable F1 (dev): 71.40
- Unsupported-answer rate (dev): 34.84
- Abstain F1 (dev): 71.17
- Supported-answer rate (among answers, dev): 62.40
- Answer rate (dev): 58.38
- Selected keep threshold: 0.65
- Max unsupported-answer rate budget: 20.00
- Output folder path: /content/drive/MyDrive/KeelNet/artifacts/stage5_colab/naufal-stage5-v1
- Executed notebook archive target: /content/drive/MyDrive/KeelNet/artifacts/stage5_colab/naufal-stage5-v1/executed-notebook/naufal-stage5-v1-executed.ipynb
- Drive mirror path: /content/drive/MyDrive/KeelNet/artifacts/stage5_colab/naufal-stage5-v1

Saved share note: /content/drive/MyDrive/KeelNet/artifacts/stage5_colab/naufal-stage5-v1/collab-share-note.md
Saved completion marker: 